<a href="https://colab.research.google.com/github/astrohimanshu/parkinsons-severity-predictor/blob/main/Parkinsons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
df_pd=pd.read_csv('/content/parkinsons_patients.csv')
df_ac_pd=pd.read_csv('/content/parkinson3features.csv')
df_hel=pd.read_csv('/content/healthy_controls.csv')
df_hel_ac=pd.read_csv('/content/health3features.csv')
print("Columns in df_pd:")
print(df_pd.columns)
print("\nColumns in df_ac_pd:")
print(df_ac_pd.columns)

In [ ]:
df_ac_pd.head()

In [ ]:
import pandas as pd


df_pd['Status'] = 0
df_ac_pd['Status'] = 0


df_parkinson = pd.concat([df_pd, df_ac_pd.drop(columns=['Status'])], axis=1)


if df_parkinson.isnull().values.any():
    print(" Warning: NaNs detected! Your indices might not be aligned.")
    # Reset indices if they were different
    df_pd = df_pd.reset_index(drop=True)
    df_ac_pd = df_ac_pd.reset_index(drop=True)
    df_parkinson = pd.concat([df_pd, df_ac_pd.drop(columns=['Status'])], axis=1)

print(df_parkinson.head())

In [ ]:
print("Columns before dropping 'Filename':")
print(df_parkinson.columns)
df_parkinson.drop('Filename',axis=1, inplace=True)
print("Columns after dropping 'Filename':")
print(df_parkinson.columns)

In [ ]:
df_hel['Status'] = 1
df_hel_ac['Status'] = 1

df_healthy = pd.concat([df_hel, df_hel_ac.drop(columns=['Status'])], axis=1)

if df_healthy.isnull().values.any():
    print(" Warning: NaNs detected! Your indices might not be aligned.")


    df_hel = df_hel.reset_index(drop=True)
    df_hel_ac = df_hel_ac.reset_index(drop=True)
    df_healthy = pd.concat([df_hel, df_hel_ac.drop(columns=['Status'])], axis=1)

print(df_healthy.head())

In [ ]:
df_final=pd.concat([df_parkinson,df_healthy],axis=0)
df_final.head()
df_final.drop('Filename', axis=1, inplace=True)

In [ ]:
df_final.head()

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

# Data Preprocessing
df = df_final.copy()


le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])


for col in df.columns:
    if df[col].dtype == 'float64' or df[col].dtype == 'int64':
        df[col] = df[col].fillna(df[col].mean())

# Approach 1: Predict Parkinson's disease status using age and sex
print("--- Approach 1: Using Age and Sex ---")
X1 = df[['age', 'sex']]
y = df['Status']

X_train1, X_test1, y_train, y_test = train_test_split(X1, y, test_size=0.3, random_state=42)

scaler1 = StandardScaler()
X_train_scaled1 = scaler1.fit_transform(X_train1)
X_test_scaled1 = scaler1.transform(X_test1)

results1 = {}

# Logistic Regression
start_time = time.time()
lr_model1 = LogisticRegression(random_state=42)
lr_model1.fit(X_train_scaled1, y_train)
lr_pred1 = lr_model1.predict(X_test_scaled1)
end_time = time.time()
results1['Logistic Regression'] = {'F1 Score': f1_score(y_test, lr_pred1), 'Time': end_time - start_time}

# SVM
start_time = time.time()
svm_model1 = SVC(random_state=42)
svm_model1.fit(X_train_scaled1, y_train)
svm_pred1 = svm_model1.predict(X_test_scaled1)
end_time = time.time()
results1['SVM'] = {'F1 Score': f1_score(y_test, svm_pred1), 'Time': end_time - start_time}

# XGBoost
start_time = time.time()
xgb_model1 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model1.fit(X_train_scaled1, y_train)
xgb_pred1 = xgb_model1.predict(X_test_scaled1)
end_time = time.time()
results1['XGBoost'] = {'F1 Score': f1_score(y_test, xgb_pred1), 'Time': end_time - start_time}

# PyTorch Neural Network
class SimpleNN(nn.Module):
    def __init__(self, input_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 10)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

start_time = time.time()
input_size1 = X_train_scaled1.shape[1]
nn_model1 = SimpleNN(input_size1)
criterion = nn.BCELoss()
optimizer = optim.Adam(nn_model1.parameters(), lr=0.001)

X_train_tensor1 = torch.tensor(X_train_scaled1, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor1 = torch.tensor(X_test_scaled1, dtype=torch.float32)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = nn_model1(X_train_tensor1)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

nn_model1.eval()
with torch.no_grad():
    nn_outputs1 = nn_model1(X_test_tensor1)
    nn_pred1 = (nn_outputs1 > 0.5).int().numpy()
end_time = time.time()
results1['PyTorch NN'] = {'F1 Score': f1_score(y_test, nn_pred1), 'Time': end_time - start_time}

print("Results for Approach 1:")
for model, metrics in results1.items():
    print(f"{model}: F1 Score = {metrics['F1 Score']:.4f}, Time = {metrics['Time']:.4f} seconds")

# Approach 2: Predict Parkinson's disease status using only audio features
print("\n--- Approach 2: Using Audio Features Only ---")

audio_features = ['Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP',
                  'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11',
                  'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'PPE']

X2 = df[audio_features]

X_train2, X_test2, _, _ = train_test_split(X2, y, test_size=0.3, random_state=42)

scaler2 = StandardScaler()
X_train_scaled2 = scaler2.fit_transform(X_train2)
X_test_scaled2 = scaler2.transform(X_test2)

results2 = {}

# Logistic Regression
start_time = time.time()
lr_model2 = LogisticRegression(random_state=42)
lr_model2.fit(X_train_scaled2, y_train)
lr_pred2 = lr_model2.predict(X_test_scaled2)
end_time = time.time()
results2['Logistic Regression'] = {'F1 Score': f1_score(y_test, lr_pred2), 'Time': end_time - start_time}

# SVM
start_time = time.time()
svm_model2 = SVC(random_state=42)
svm_model2.fit(X_train_scaled2, y_train)
svm_pred2 = svm_model2.predict(X_test_scaled2)
end_time = time.time()
results2['SVM'] = {'F1 Score': f1_score(y_test, svm_pred2), 'Time': end_time - start_time}

# XGBoost
start_time = time.time()
xgb_model2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model2.fit(X_train_scaled2, y_train)
xgb_pred2 = xgb_model2.predict(X_test_scaled2)
end_time = time.time()
results2['XGBoost'] = {'F1 Score': f1_score(y_test, xgb_pred2), 'Time': end_time - start_time}

# PyTorch Neural Network
start_time = time.time()
input_size2 = X_train_scaled2.shape[1]
nn_model2 = SimpleNN(input_size2)
criterion = nn.BCELoss()
optimizer = optim.Adam(nn_model2.parameters(), lr=0.001)

X_train_tensor2 = torch.tensor(X_train_scaled2, dtype=torch.float32)
X_test_tensor2 = torch.tensor(X_test_scaled2, dtype=torch.float32)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = nn_model2(X_train_tensor2)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

nn_model2.eval()
with torch.no_grad():
    nn_outputs2 = nn_model2(X_test_tensor2)
    nn_pred2 = (nn_outputs2 > 0.5).int().numpy()
end_time = time.time()
results2['PyTorch NN'] = {'F1 Score': f1_score(y_test, nn_pred2), 'Time': end_time - start_time}

print("Results for Approach 2:")
for model, metrics in results2.items():
    print(f"{model}: F1 Score = {metrics['F1 Score']:.4f}, Time = {metrics['Time']:.4f} seconds")

# Find model with least time in Approach 1
best_model_time1 = min(results1, key=lambda k: results1[k]['Time'])
print(f"\nApproach 1: Model with least time is {best_model_time1} with time {results1[best_model_time1]['Time']:.4f} seconds")

# Find model with least time in Approach 2
best_model_time2 = min(results2, key=lambda k: results2[k]['Time'])
print(f"Approach 2: Model with least time is {best_model_time2} with time {results2[best_model_time2]['Time']:.4f} seconds")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_final['sex'] = le.fit_transform(df_final['sex'])




corr_matrix = df_final.corr()
print(corr_matrix)

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix[['Status']].sort_values(by='Status', ascending=False),
            annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Feature Correlation with Parkinson's Status")
plt.show()

SVM with PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score


jitter_cols = ['Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP']
shimmer_cols = ['Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA']
strong_acoustic = ['RPDE', 'DFA', 'PPE', 'NHR', 'HNR']


scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_final[jitter_cols + shimmer_cols + strong_acoustic])
df_scaled = pd.DataFrame(df_scaled, columns=jitter_cols + shimmer_cols + strong_acoustic)


pca = PCA(n_components=2)
instability_features = pca.fit_transform(df_scaled[jitter_cols + shimmer_cols])
df_pca = pd.DataFrame(instability_features, columns=['Instability_PC1', 'Instability_PC2'])


X = pd.concat([df_pca, df_scaled[strong_acoustic]], axis=1)
y = df_final['Status']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


model = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')
model.fit(X_train, y_train)

# Results
y_pred = model.predict(X_test)
print(f"🚀 New Voice-Only F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(classification_report(y_test, y_pred))

Conclusion:Not advisable as it dropped further

Feature Engineering

In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif

# Non Linear Scaler to force data to fit into Normal distribution
scaler = QuantileTransformer(output_distribution='normal', n_quantiles=100)
X_raw = df_final.drop(columns=['Status', 'age', 'sex'], errors='ignore')
X_scaled = scaler.fit_transform(X_raw)

# K Best features
selector = SelectKBest(score_func=f_classif, k=10)
X_best = selector.fit_transform(X_scaled, df_final['Status'])


mask = selector.get_support()
chosen_features = X_raw.columns[mask]
print(f"Selected Features: {list(chosen_features)}")


X_train, X_test, y_train, y_test = train_test_split(X_best, df_final['Status'],
                                                    test_size=0.2, random_state=42, stratify=df_final['Status'])


rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# Results
y_pred = rf_model.predict(X_test)
print(f"Optimized Voice-Only F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(classification_report(y_test, y_pred))

SMOTE+XGB

In [ ]:
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import RobustScaler


X = df_final.drop(columns=['Status', 'age', 'sex', 'Filename'], errors='ignore')
y = df_final['Status']

#Robust Scaling
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2,
                                                    random_state=42, stratify=y)


sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"📈 Resampled Training Set Size: {len(X_train_res)} (was {len(X_train)})")

# XGBoost
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False
)

model.fit(X_train_res, y_train_res)

# Results
y_pred = model.predict(X_test)
print(f"Optimized Voice-Only F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(classification_report(y_test, y_pred))




Demorgraphy Wise partition to check robustness

In [ ]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import RobustScaler


X_raw = df_final.drop(columns=['Status', 'Filename'], errors='ignore')
y = df_final['Status']


indices = np.arange(len(df_final))
X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    indices, y, test_size=0.2, random_state=42, stratify=y
)


voice_cols = [c for c in X_raw.columns if c not in ['age', 'sex']]
X_voice = X_raw[voice_cols]


scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_voice.iloc[X_train_idx])
X_test_scaled = scaler.transform(X_voice.iloc[X_test_idx])

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)


model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss'
)
model.fit(X_train_res, y_train_res)


y_pred = model.predict(X_test_scaled)


eval_df = df_final.iloc[X_test_idx].copy()
eval_df['y_true'] = y_test.values
eval_df['y_pred'] = y_pred

eval_df['age_group'] = pd.cut(eval_df['age'], bins=[0, 50, 65, 100], labels=['<50', '51-65', '66+'])

print(f" Overall F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}\n")

def print_partition(df, group_col, label):
    print(f"--- {label} Partition ---")
    for group in df[group_col].unique():
        subset = df[df[group_col] == group]
        if len(subset) > 0:
            score = f1_score(subset['y_true'], subset['y_pred'], average='weighted')
            print(f"Group: {group: <8} | F1: {score:.4f} | Samples: {len(subset)}")
    print("-" * 30)


print_partition(eval_df, 'sex', 'Sex (0=F, 1=M)')
print_partition(eval_df, 'age_group', 'Age')

Stacked Ensemble

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score


X = df_final.drop(columns=['Status', 'age', 'sex', 'Filename'], errors='ignore')
y = df_final['Status']

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2,
                                                    random_state=42, stratify=y)


base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('svc', SVC(kernel='rbf', probability=True, class_weight='balanced')),
    ('xgb', xgb.XGBClassifier(learning_rate=0.05, n_estimators=100, use_label_encoder=False, eval_metric='logloss'))
]


meta_model = LogisticRegression()


stack_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

stack_model.fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
print(f" Stacked Ensemble F1 Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(classification_report(y_test, y_pred))

Front end using Gradio

In [ ]:
import pandas as pd
import numpy as np
import gradio as gr
import xgboost as xgb
import re
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

# 1. SETUP: Train the 0.88 F1 Model

X_voice = df_final.drop(columns=['Status', 'age', 'sex', 'Filename'], errors='ignore')
y = df_final['Status']
feature_names = X_voice.columns.tolist()

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_voice)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_scaled, y)

demo_model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05)
demo_model.fit(X_res, y_res)

# 2. THE PREDICTION & PREVIEW LOGIC
def process_and_predict(input_text):
    try:
        # Regex to find all numbers: handles commas, spaces, tabs, brackets
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", input_text)
        data = [float(n) for n in numbers]

        if not data:
            return "❌ No data found.", pd.DataFrame(), "Please paste a valid row."


        trimmed_data = data[:len(feature_names)]

        # Creating a Preview Table for the first 5 features
        preview_data = {
            "Feature Name": feature_names[:5] + ["..."],
            "Value Extracted": [round(x, 6) for x in trimmed_data[:5]] + ["..."]
        }
        preview_df = pd.DataFrame(preview_data)

        if len(trimmed_data) < len(feature_names):
            return "❌ Missing Features", preview_df, f"Error: Found {len(trimmed_data)} numbers, but model needs {len(feature_names)}."

        # Scale and Predict
        input_array = np.array(trimmed_data).reshape(1, -1)
        scaled_data = scaler.transform(input_array)
        prediction = demo_model.predict(scaled_data)[0]
        probs = demo_model.predict_proba(scaled_data)[0]

        label = "🔴 PARKINSON'S (Status 1)" if prediction == 1 else "🟢 HEALTHY (Status 0)"
        confidence = probs[1] if prediction == 1 else probs[0]

        return label, preview_df, f"Analysis Complete. Confidence: {confidence*100:.2f}%"

    except Exception as e:
        return "⚠️ Error", pd.DataFrame(), str(e)

# 3. GRADIO UI DESIGN
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Parkinson's Acoustic Diagnostic Demo")
    gr.Markdown("### Instructions:\n1. Copy a full row from your CSV (including commas or spaces).\n2. Paste it below.\n3. Verify the 'Data Preview' table to ensure the numbers align.")

    with gr.Row():
        with gr.Column(scale=2):
            input_box = gr.Textbox(
                label="Paste Data Row Here",
                placeholder="0.012, 0.0007, 1.234...",
                lines=4
            )
            submit_btn = gr.Button("🔍 Run Diagnostic Analysis", variant="primary")

        with gr.Column(scale=1):
            preview_table = gr.DataFrame(label="Data Preview (First 5 Features)")

    with gr.Row():
        result_display = gr.Label(label="Prediction Result")
        status_msg = gr.Textbox(label="System Status", interactive=False)

    submit_btn.click(
        fn=process_and_predict,
        inputs=input_box,
        outputs=[result_display, preview_table, status_msg]
    )

    gr.Examples(
        examples=[[", ".join(map(str, X_voice.iloc[0].values.round(6)))]],
        inputs=input_box,
        label="Demo Sample (Click to Load):"
    )

demo.launch(share=True)